In [1]:
import gc
import os
import sqlite3

import numpy as np
import pandas as pd
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from dataset.pac_matrix import PAC5MatrixDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


k_folds = 5
target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]
dataset = "PAC5-4"
model_path_template = f"../models/gcnn_1.1M-pac_grid-[{dataset}]-fold_{{fold}}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = f"gcnn_1.1M-pac_grid-[{dataset}]-fold_{{fold}}"
num_tasks = len(target_properties)
loss_weights = torch.ones(num_tasks, device=device)

W0721 23:01:22.557000 31432 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [3]:
fold_results = []

for fold in range(1, k_folds + 1):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    val_path = model_dataset_dir + f"/{dataset}/fold_{fold}_val.csv"

    train_dataset = PAC5MatrixDataset(
        train_path, properties=target_properties, num_workers=12
    )
    val_dataset = PAC5MatrixDataset(
        val_path, properties=target_properties, num_workers=12
    )

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    val_dataset.targets = torch.tensor(
        scaler.transform(val_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        property_names=target_properties,
        scaler=scaler,
        enable_progress_bar=False,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_dataset, val_dataset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 2/5 ]======


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ GroupEquivariantCNN │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss             │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict          │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict          │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict          │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict          │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict          │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict          │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict          │      0 │ train │     0 │
└───┴────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.365                                                                      
Modules in train mode: 86                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


...

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.365                                                                      
Modules in train mode: 86                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 2.55022 | xtb_homo Val RMSD: 0.87085 | xtb_lumo Val RMSD: 0.85854 | xtb_hl_gap Val RMSD: 1.07607 | pm7_homo Val RMSD: 1.18135 | pm7_lumo Val RMSD: 0.56949 | pm7_hl_gap Val RMSD: 2.04380 | pm7_hof Val RMSD: 759.91925
Epoch 002 | Scaled Train Loss: 0.50014 | Scaled Val Loss: 0.30155 | xtb_homo Val RMSD: 0.43130 | xtb_lumo Val RMSD: 0.54009 | xtb_hl_gap Val RMSD: 0.41749 | pm7_homo Val RMSD: 0.31170 | pm7_lumo Val RMSD: 0.34697 | pm7_hl_gap Val RMSD: 0.43298 | pm7_hof Val RMSD: 193.85220
Epoch 003 | Scaled Train Loss: 0.28199 | Scaled Val Loss: 0.15524 | xtb_homo Val RMSD: 0.32586 | xtb_lumo Val RMSD: 0.41380 | xtb_hl_gap Val RMSD: 0.23504 | pm7_homo Val RMSD: 0.34703 | pm7_lumo Val RMSD: 0.27055 | pm7_hl_gap Val RMSD: 0.25644 | pm7_hof Val RMSD: 108.36427
Epoch 004 | Scaled Train Loss: 0.19354 | Scaled Val Loss: 0.12951 | xtb_homo Val RMSD: 0.34025 | xtb_lumo Val RMSD: 0.36065 | xtb_hl_gap Val RMSD: 0.21720 | pm7_homo Val RMSD: 0.2840

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00533 | Scaled Val Loss: 0.00683 | xtb_homo Val RMSD: 0.05690 | xtb_lumo Val RMSD: 0.05895 | xtb_hl_gap Val RMSD: 0.07677 | pm7_homo Val RMSD: 0.06888 | pm7_lumo Val RMSD: 0.04880 | pm7_hl_gap Val RMSD: 0.08083 | pm7_hof Val RMSD: 14.35597
Best scaled validation loss for fold 5: 0.0068

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0063, 0.011, 0.0074, 0.0068]
Average Scaled Loss: 0.0079 ± 0.0019


Epoch  72 | Scaled Train Loss: 0.0029 | Scaled Val Loss: 0.0106 | xtb_homo Val RMSD: 0.0369 | xtb_lumo Val RMSD:
0.0418 | xtb_hl_gap Val RMSD: 0.0524 | pm7_homo Val RMSD: 0.0498 | pm7_lumo Val RMSD: 0.0416 | pm7_hl_gap Val
RMSD: 0.0624 | pm7_hof Val RMSD: 16.1357\
Epoch   4 | Scaled Train Loss: 0.0019 | Scaled Val Loss: 0.0103 | xtb_homo Val RMSD: 0.0366 | xtb_lumo Val RMSD:
0.0411 | xtb_hl_gap Val RMSD: 0.0519 | pm7_homo Val RMSD: 0.0494 | pm7_lumo Val RMSD: 0.0411 | pm7_hl_gap Val
RMSD: 0.0618 | pm7_hof Val RMSD: 16.1258

Epoch  91 | Scaled Train Loss: 0.0020 | Scaled Test Loss: 0.0090 | xtb_homo RMSD: 0.0348 | xtb_lumo RMSD: 0.0383 | xtb_hl_gap RMSD: 0.0497 | pm7_homo RMSD: 0.0498 | pm7_lumo RMSD: 0.0403 | pm7_hl_gap RMSD: 0.0608 | pm7_hof RMSD: 12.3982

In [2]:
import numpy as np

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in range(1, k_folds + 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"

    train_dataset = PAC5MatrixDataset(
        train_path, properties=target_properties, num_workers=12
    )
    test_dataset = PAC5MatrixDataset(
        test_path, properties=target_properties, num_workers=12
    )

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    test_dataset.targets = torch.tensor(
        scaler.transform(test_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    intercepted_preds = []

    def capture_predictions(module, input, output):
        intercepted_preds.append(output.detach().cpu().numpy())

    hook_handle = model.register_forward_hook(capture_predictions)

    print("\n======[ Train Set Evaluation ]======")
    intercepted_preds.clear()
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]
    train_preds_scaled = np.vstack(intercepted_preds)

    print("\n======[ Test Set Evaluation ]======")
    intercepted_preds.clear()
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]
    test_preds_scaled = np.vstack(intercepted_preds)

    hook_handle.remove()

    # Convert predictions back to original units
    train_preds_unscaled = scaler.inverse_transform(train_preds_scaled)
    test_preds_unscaled = scaler.inverse_transform(test_preds_scaled)

    # Convert targets back to original units
    train_targets_unscaled = scaler.inverse_transform(train_dataset.targets.numpy())
    test_targets_unscaled = scaler.inverse_transform(test_dataset.targets.numpy())

    train_df = pd.DataFrame({"smiles": pd.read_csv(train_path)["smiles"]})
    test_df = pd.DataFrame({"smiles": pd.read_csv(test_path)["smiles"]})

    for i, prop in enumerate(target_properties):
        train_df[f"{prop}"] = train_preds_unscaled[:, i]
        test_df[f"{prop}"] = test_preds_unscaled[:, i]

    train_csv_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train_predictions.csv"
    test_csv_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test_predictions.csv"

    train_df.to_csv(train_csv_path, index=False)
    test_df.to_csv(test_csv_path, index=False)
    print("\n======[ Saved Predictions ]======")
    print(f" + {train_csv_path}")
    print(f" + {test_csv_path}\n")

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    del model
    del lightning_model
    del trainer
    del scaler
    del train_results
    del test_results
    del train_loader
    del test_loader
    del train_dataset
    del test_dataset
    del train_df
    del test_df
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")

    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )


======[ Evaluating fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` 


======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0106	RMSD: 0.0141	Max: 0.1513	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0296	RMSD: 0.0569	Max: 1.0444	Min: 0.0000	R^2: 0.9951
======[ xtb_lumo ]======
 + Train -> MAE: 0.0111	RMSD: 0.0145	Max: 0.1393	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0287	RMSD: 0.0512	Max: 0.8527	Min: 0.0000	R^2: 0.9963
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0086	RMSD: 0.0117	Max: 0.1496	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0369	RMSD: 0.0735	Max: 0.9484	Min: 0.0000	R^2: 0.9860
======[ pm7_homo ]======
 + Train -> MAE: 0.0109	RMSD: 0.0145	Max: 0.1310	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0328	RMSD: 0.0689	Max: 1.6943	Min: 0.0000	R^2: 0.9936
======[ pm7_lumo ]======
 + Train -> MAE: 0.0078	RMSD: 0.0109	Max: 0.1331	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0238	RMSD: 0.0459	Max: 0.6997	Min: 0.0000	R^2: 0.9936
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0108	RMSD: 0.0142	Max: 0.1183	Min: 0.0000	R^2: 0.9997
 + Test -> 

...


======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0095	RMSD: 0.0129	Max: 0.1851	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0272	RMSD: 0.0484	Max: 0.4670	Min: 0.0000	R^2: 0.9965
======[ xtb_lumo ]======
 + Train -> MAE: 0.0133	RMSD: 0.0168	Max: 0.1670	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0298	RMSD: 0.0527	Max: 0.6375	Min: 0.0000	R^2: 0.9962
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0130	RMSD: 0.0161	Max: 0.1023	Min: 0.0000	R^2: 0.9993
 + Test -> MAE: 0.0381	RMSD: 0.0699	Max: 0.9264	Min: 0.0001	R^2: 0.9876
======[ pm7_homo ]======
 + Train -> MAE: 0.0103	RMSD: 0.0141	Max: 0.1891	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0344	RMSD: 0.0764	Max: 1.9325	Min: 0.0000	R^2: 0.9920
======[ pm7_lumo ]======
 + Train -> MAE: 0.0107	RMSD: 0.0138	Max: 0.1618	Min: 0.0000	R^2: 0.9994
 + Test -> MAE: 0.0270	RMSD: 0.0672	Max: 2.0457	Min: 0.0000	R^2: 0.9868
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0103	RMSD: 0.0135	Max: 0.1499	Min: 0.0000	R^2: 0.9997
 + Test -> 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` 


======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0124	RMSD: 0.0157	Max: 0.2010	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0295	RMSD: 0.0546	Max: 0.9051	Min: 0.0000	R^2: 0.9953
======[ xtb_lumo ]======
 + Train -> MAE: 0.0104	RMSD: 0.0136	Max: 0.1463	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0297	RMSD: 0.0536	Max: 0.6047	Min: 0.0000	R^2: 0.9959
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0093	RMSD: 0.0124	Max: 0.1552	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0377	RMSD: 0.0751	Max: 1.1366	Min: 0.0000	R^2: 0.9857
======[ pm7_homo ]======
 + Train -> MAE: 0.0114	RMSD: 0.0148	Max: 0.1197	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0347	RMSD: 0.0683	Max: 0.9151	Min: 0.0000	R^2: 0.9933
======[ pm7_lumo ]======
 + Train -> MAE: 0.0077	RMSD: 0.0104	Max: 0.1327	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0268	RMSD: 0.0542	Max: 1.0214	Min: 0.0000	R^2: 0.9910
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0106	RMSD: 0.0137	Max: 0.1089	Min: 0.0000	R^2: 0.9997
 + Test -> 

In [5]:
import numpy as np

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in range(1, 2):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    # train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    # test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"
    # #
    # train_dataset = PAC5MatrixDataset(train_path, properties=target_properties, num_workers=12)
    # test_dataset = PAC5MatrixDataset(test_path, properties=target_properties, num_workers=12)
    # #
    # scaler = MultiTargetScaler()
    # scaler.fit(train_dataset.targets.numpy())
    # #
    # train_dataset.targets = torch.tensor(scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32)
    # test_dataset.targets = torch.tensor(scaler.transform(test_dataset.targets.numpy()), dtype=torch.float32)
    # #
    # train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
    # test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=16
    )
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    intercepted_preds = []

    def capture_predictions(module, input, output):
        intercepted_preds.append(output.detach().cpu().numpy())

    hook_handle = model.register_forward_hook(capture_predictions)

    print("\n======[ Train Set Evaluation ]======")
    intercepted_preds.clear()
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]
    train_preds_scaled = np.vstack(intercepted_preds)

    print("\n======[ Test Set Evaluation ]======")
    intercepted_preds.clear()
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]
    test_preds_scaled = np.vstack(intercepted_preds)

    hook_handle.remove()

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    del model
    del lightning_model
    del trainer
    del scaler
    del train_results
    del test_results
    del train_loader
    del test_loader
    del train_dataset
    del test_dataset
    # del train_df
    # del test_df
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")

    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Evaluating fold 1/5 ]======

======[ Train Set Evaluation ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0202	RMSD: 0.0275	Max: 0.6156	Min: 0.0000	R^2: 0.9959
 + Test -> MAE: 0.0274	RMSD: 0.0425	Max: 0.7061	Min: 0.0000	R^2: 0.9903
======[ xtb_lumo ]======
 + Train -> MAE: 0.0221	RMSD: 0.0298	Max: 0.6451	Min: 0.0000	R^2: 0.9969
 + Test -> MAE: 0.0292	RMSD: 0.0445	Max: 1.1477	Min: 0.0000	R^2: 0.9930
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0235	RMSD: 0.0317	Max: 0.4610	Min: 0.0000	R^2: 0.9941
 + Test -> MAE: 0.0360	RMSD: 0.0577	Max: 1.6969	Min: 0.0000	R^2: 0.9804
======[ pm7_homo ]======
 + Train -> MAE: 0.0258	RMSD: 0.0356	Max: 0.4523	Min: 0.0000	R^2: 0.9951
 + Test -> MAE: 0.0360	RMSD: 0.0549	Max: 1.0783	Min: 0.0000	R^2: 0.9883
======[ pm7_lumo ]======
 + Train -> MAE: 0.0206	RMSD: 0.0280	Max: 0.3428	Min: 0.0000	R^2: 0.9936
 + Test -> MAE: 0.0294	RMSD: 0.0458	Max: 2.8561	Min: 0.0000	R^2: 0.9828
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0279	RMSD: 0.0384	Max: 0.4695	Min: 0.0000	R^2: 0.9928
 + Test -> 

D:\Coding\pac-gcnn\.venv\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
D:\Coding\pac-gcnn\.venv\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [8]:
pred_df

,xtb_homo_true,xtb_homo_pred,xtb_lumo_true,xtb_lumo_pred,xtb_hl_gap_true,xtb_hl_gap_pred,pm7_homo_true,pm7_homo_pred,pm7_lumo_true,pm7_lumo_pred,pm7_hl_gap_true,pm7_hl_gap_pred,pm7_hof_true,pm7_hof_pred
0,-11.0922,-11.083380,-8.1036,-8.084236,2.988553,2.996783,-9.459766,-9.452770,-1.331181,-1.389217,8.128585,8.061984,-206.031250,-211.982060
1,-11.0804,-11.091131,-8.2475,-8.262691,2.832883,2.826418,-9.296226,-9.332479,-1.431863,-1.423617,7.864363,7.907770,-259.608220,-273.496460
2,-9.3417,-9.355114,-7.7947,-7.783248,1.547056,1.571241,-9.183843,-9.194277,-1.002195,-1.001607,8.181647,8.192618,288.583370,292.660640
3,-9.3477,-9.357862,-5.1763,-5.411232,4.171439,3.942137,-8.750093,-8.590299,0.623141,0.475728,9.373234,9.065076,149.048700,158.296480
4,-11.4976,-11.485188,-8.0847,-8.103149,3.412844,3.379431,-10.367266,-10.359633,-1.464517,-1.476285,8.902749,8.880926,155.672060,136.714630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135740,-9.6568,-9.680234,-6.6968,-6.720396,2.959934,2.958658,-8.430632,-8.464740,-0.392660,-0.389320,8.037971,8.075181,232.355160,226.781770
135741,-9.8175,-9.857690,-6.7725,-6.798913,3.045009,3.057937,-8.474170,-8.471584,-0.412525,-0.412879,8.061645,8.058105,36.390564,33.151520
135742,-9.9139,-9.923721,-7.2949,-7.248040,2.618972,2.675186,-8.556892,-8.558799,-0.763824,-0.700411,7.793069,7.857897,362.707920,363.655200
135743,-9.5791,-9.591671,-6.3638,-6.409130,3.215231,3.180915,-8.294847,-8.293131,-0.114288,-0.109021,8.180559,8.183658,194.925170,185.374730


In [9]:
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dataset.common import DATA_PAC5_SQLITE

dataset_dirs = [
    ("PAC5-3", "PAC5-3"),
    ("PAC5-2", "PAC5-2"),
    ("PAC5-1", "PAC5-1"),
]

colors = {
    "PAC5-3": px.colors.qualitative.Plotly[0],
    "PAC5-2": px.colors.qualitative.Plotly[1],
    "PAC5-1": px.colors.qualitative.Plotly[2],
}

marker_symbols = {"PAC5-3": "circle", "PAC5-2": "diamond", "PAC5-1": "square"}

target_prop = "pm7_hl_gap"
k_folds = 5

conn = sqlite3.connect(DATA_PAC5_SQLITE)
c = conn.cursor()
c.execute(
    "SELECT smiles, category FROM pac5 WHERE pm7_hl_gap IS NOT NULL AND is_open_shell = 0"
)
all_rows = c.fetchall()
conn.close()

cat_map = {s: cat for s, cat in all_rows}

rmsd_data = {label: {} for _, label in dataset_dirs}
count_data = {label: {} for _, label in dataset_dirs}

for dir_name, label in dataset_dirs:
    for fold in range(1, k_folds + 1):
        true_path = f"../dataset/{dir_name}/fold_{fold}_test.csv"
        pred_path = f"../dataset/{dir_name}/fold_{fold}_test_predictions.csv"

        true_df = pd.read_csv(true_path)
        pred_df = pd.read_csv(pred_path)

        if len(true_df) != len(pred_df):
            raise ValueError(
                f"Row count mismatch in fold {fold} of {dir_name}: {len(true_df)} test rows, {len(pred_df)} prediction rows"
            )

        # Rows matched by position; SMILES column from test file only
        pred_df = pred_df.drop(columns="smiles", errors="ignore")
        df = true_df.join(pred_df, lsuffix="_true", rsuffix="_pred")
        df["category"] = df["smiles"].map(cat_map)
        df = df.dropna(subset=["category"])

        for cat, group in df.groupby("category"):
            y_true = group[f"{target_prop}_true"].to_numpy()
            y_pred = group[f"{target_prop}_pred"].to_numpy()
            fold_rmsd = np.sqrt(np.mean((y_pred - y_true) ** 2))

            if cat not in rmsd_data[label]:
                rmsd_data[label][cat] = []
            rmsd_data[label][cat].append(fold_rmsd)

            if cat not in count_data[label]:
                count_data[label][cat] = 0
            count_data[label][cat] += len(group)

agg_data = []
for _, label in dataset_dirs:
    for cat, fold_rmsds in rmsd_data[label].items():
        mean_rmsd = np.mean(fold_rmsds)
        std_rmsd = np.std(fold_rmsds, ddof=1) if len(fold_rmsds) > 1 else 0.0

        agg_data.append(
            {
                "dataset": label,
                "category": str(cat),
                "mean_rmsd": mean_rmsd,
                "std_rmsd": std_rmsd,
                "count": count_data[label][cat],
            }
        )

agg_df = pd.DataFrame(agg_data)

pac5_3_df = agg_df[agg_df["dataset"] == "PAC5-3"].sort_values(
    by="mean_rmsd", ascending=False
)
sorted_categories = pac5_3_df["category"].tolist()

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Numeric x-axis scale for precise scatter shifting
x_base = np.arange(len(sorted_categories))
bar_width = 0.8 / len(
    dataset_dirs
)  # 0.8 = default Plotly group width per tick (1 - bargap)
offsets = [
    (i - (len(dataset_dirs) - 1) / 2) * bar_width for i in range(len(dataset_dirs))
]

for i, (dir_name, label) in enumerate(dataset_dirs):
    ds_df = (
        agg_df[agg_df["dataset"] == label]
        .set_index("category")
        .reindex(sorted_categories)
        .reset_index()
    )

    fig.add_trace(
        go.Bar(
            x=x_base,
            y=ds_df["mean_rmsd"],
            name=f"{label} (Bars)",
            legendgroup=label,
            marker_color=colors[label],
            error_y=dict(
                type="data",
                array=ds_df["std_rmsd"],
                color="black",
                thickness=1,
                width=3,
            ),
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scattergl(
            x=x_base
            + offsets[
                i
            ],  # Shift scatter markers to center of each bar
            y=ds_df["count"],
            name=f"{label} (Markers)",
            legendgroup=label,
            mode="markers",
            marker=dict(
                symbol=marker_symbols[label],
                color=colors[label],
                size=8,
                line=dict(color="black", width=1),
            ),
        ),
        secondary_y=True,
    )

fig.update_layout(
    template="plotly",
    barmode="group",
    bargap=0.2,  # Enforce 0.2 gap for explicit math matching
    bargroupgap=0.0,  # Enforce 0.0 group gap for explicit math matching
    font=dict(size=20),
    legend=dict(
        title="<b>Dataset</b>",
        orientation="h",
        traceorder="grouped",
        yanchor="bottom",
        y=1.04,
        xanchor="center",
        x=0.48,
        bordercolor="black",
        borderwidth=1,
    ),
    margin=dict(t=100),
)

fig.update_xaxes(
    title_text="<b>Category</b>",
    tickmode="array",
    tickvals=x_base,
    ticktext=sorted_categories,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
)

fig.update_yaxes(
    title_text="<b>(Bars) PM7 HL-gap RMSD (eV)</b>",
    secondary_y=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
)

fig.update_yaxes(
    title_text="<b>(Markers) Sample Count</b>",
    secondary_y=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    tickformat="~s",
)
fig.update_layout(width=1200, height=600)
fig.write_image("../figures/manuscript_figure_4.png", scale=2)
fig.show()